### Assignment
Train a word2vec model on the [Sports & Outdoors Reviews Dataset](http://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Sports_and_Outdoors_5.json.gz) Once you train a model on this, find the words most similar to 'awful' and find similarities between the following word tuples: ('good', 'great'), ('slow','steady')

https://www.youtube.com/watch?v=Q2NtCcqmIww

In [ ]:
!pip install gensim
!pip install python-levenshtein

In [4]:
import gensim
import pandas as pd
from urllib.request import urlretrieve
import gzip
import shutil

Reading and Exploring the Dataset

Link to the dataset: http://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Sports_and_Outdoors_5.json.gz

In [5]:
url = "http://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Sports_and_Outdoors_5.json.gz"
gz_filename = "reviews_Sports_and_Outdoors_5.json.gz"
json_filename = "reviews_Sports_and_Outdoors_5.json"

urlretrieve(url, gz_filename)

with gzip.open(gz_filename, 'rt') as f:
    with open(json_filename, 'w') as out_f:
        shutil.copyfileobj(f, out_f)

sports_reviews = pd.read_json(json_filename, lines=True)

sports_reviews.head()

,reviewerID,asin,reviewerName,helpful,reviewText,overall,summary,unixReviewTime,reviewTime
0,AIXZKN4ACSKI,1881509818,David Briner,"[0, 0]",This came in on time and I am veru happy with ...,5,Woks very good,1390694400,"01 26, 2014"
1,A1L5P841VIO02V,1881509818,Jason A. Kramer,"[1, 1]",I had a factory Glock tool that I was using fo...,5,Works as well as the factory tool,1328140800,"02 2, 2012"
2,AB2W04NI4OEAD,1881509818,J. Fernald,"[2, 2]",If you don't have a 3/32 punch or would like t...,4,"It's a punch, that's all.",1330387200,"02 28, 2012"
3,A148SVSWKTJKU6,1881509818,"Jusitn A. Watts ""Maverick9614""","[0, 0]",This works no better than any 3/32 punch you w...,4,It's a punch with a Glock logo.,1328400000,"02 5, 2012"
4,AAAWJ6LW9WMOO,1881509818,Material Man,"[0, 0]",I purchased this thinking maybe I need a speci...,4,"Ok,tool does what a regular punch does.",1366675200,"04 23, 2013"


We are interested only in reviewText column, but we need to do some preprocessing to train Word2Vec model.
- Convert all word to lower case
- Remove puncutaions
- Remove words like a, I, the, it, etc.


### Lets take a look how this preprocessing can be done

In [6]:
sports_reviews['reviewText'][0]

'This came in on time and I am veru happy with it, I haved used it already and it makes taking out the pins in my glock 32 very easy'

In [7]:
gensim.utils.simple_preprocess(sports_reviews['reviewText'][0])

['this',
 'came',
 'in',
 'on',
 'time',
 'and',
 'am',
 'veru',
 'happy',
 'with',
 'it',
 'haved',
 'used',
 'it',
 'already',
 'and',
 'it',
 'makes',
 'taking',
 'out',
 'the',
 'pins',
 'in',
 'my',
 'glock',
 'very',
 'easy']

You can see it has broken the sentence into list of tokens and applied the preprocessing mentioned above

Now, let's do the same for entire reviewText column

In [9]:
review_text = sports_reviews['reviewText'].apply(gensim.utils.simple_preprocess)
review_text

,reviewText
0,"[this, came, in, on, time, and, am, veru, happ..."
1,"[had, factory, glock, tool, that, was, using, ..."
2,"[if, you, don, have, punch, or, would, like, t..."
3,"[this, works, no, better, than, any, punch, yo..."
4,"[purchased, this, thinking, maybe, need, speci..."
...,...
296332,"[this, is, water, bottle, done, right, it, is,..."
296333,"[if, you, re, looking, for, an, insulated, wat..."
296334,"[this, hydracentials, sporty, oz, double, insu..."
296335,"[as, usual, received, this, item, free, in, ex..."


In [10]:
model = gensim.models.Word2Vec(
    window=10,
    min_count=2,
    workers=4
)

In [11]:
model.build_vocab(review_text, progress_per=1000)

In [12]:
print(model.epochs, model.corpus_count, model.corpus_total_words)

5 296337 24299307


In [13]:
model.train(review_text, total_examples=model.corpus_count, epochs=model.epochs)

(91342616, 121496535)

In [14]:
from google.colab import files

model_path = "/content/word2vec_reviews_Sports_and_Outdoors.model"
model.save(model_path)
files.download(model_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:
# Consine similarity between two words
similarity = model.wv.similarity(w1 = 'ball', w2 = 'bat')

In [17]:
similarity

np.float32(0.6848732)

In [18]:
model.wv.similarity(w1 = 'slow', w2 = 'steady')

np.float32(0.3668859)

In [19]:
model.wv.similarity(w1 = 'good', w2 = 'great')

np.float32(0.78778934)